# SE1 vs SE2 测序深度统计分析报告

本 Notebook 用于交互式分析样本及 Target（扩增子）层面的 SE1/SE2 深度差异。

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import os

%matplotlib inline
sns.set_theme(style="whitegrid")

# 1. 加载数据
df = pd.read_csv("../SKII15275_PA_depth_processed.txt", sep='\t')
se1_cols = [c for c in df.columns if c.endswith('_Depth_SE1')]
se2_cols = [c for c in df.columns if c.endswith('_Depth_SE2')]
samples = [c.replace('_Depth_SE1', '') for c in se1_cols]

print(f"Loaded {len(samples)} samples.")

## 1. 样本层级相关性与比值分布
观察 SE1 和 SE2 是否存在系统性整体偏差。

In [ ]:
sample_stats_df = pd.read_csv("summary_by_sample.csv")

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
sns.histplot(sample_stats_df['Ratio_SE1_SE2'], bins=10, kde=True)
plt.axvline(1, color='red', linestyle='--')
plt.title('Distribution of SE1/SE2 Ratio per Sample')

plt.subplot(1, 2, 2)
sns.scatterplot(data=sample_stats_df, x='Mean_SE1', y='Mean_SE2')
max_m = max(sample_stats_df['Mean_SE1'].max(), sample_stats_df['Mean_SE2'].max())
plt.plot([0, max_m], [0, max_m], 'r--')
plt.title('Sample Mean Depth: SE1 vs SE2')
plt.show()

## 2. Target (扩增子) 层级差异分析
寻找那些在所有样本中表现异常（如 SE1 远高于 SE2）的扩增子。

In [ ]:
target_info = pd.read_csv("summary_by_target.csv")

plt.figure(figsize=(10, 8))
sns.scatterplot(data=target_info, x='Avg_SE1', y='Avg_SE2', hue='Ratio_SE1_SE2', palette='viridis', alpha=0.6)
max_t = max(target_info['Avg_SE1'].max(), target_info['Avg_SE2'].max())
plt.plot([0, max_t], [0, max_t], 'r--')
plt.xscale('log')
plt.yscale('log')
plt.title('Target-level Avg Depth (Log Scale)')
plt.show()

## 3. 筛选异常 Target (比值偏差 > 20%)
这些扩增子可能需要优化引物设计或测序参数。

In [ ]:
outliers = target_info[(target_info['Ratio_SE1_SE2'] > 1.2) | (target_info['Ratio_SE1_SE2'] < 0.8)]
print(f"找到 {len(outliers)} 个偏差较大的扩增子。")
outliers.sort_values('Ratio_SE1_SE2', ascending=False).head(10)